# Where COCO data was downloaded from:
I just chose to use the validation set because its 1GB as opposed to the 18GB training dataset
1. Download and unzip the annotations (contains both train and val JSONs)
```
wget http://images.cocodataset.org/annotations/annotations_trainval2017.zip
unzip annotations_trainval2017.zip
```

2. Download and unzip the validation images (1GB)
```
wget http://images.cocodataset.org/zips/val2017.zip
unzip val2017.zip
```

This is what i did in terminal inside my nanochat uv environment before running this notebook:
```
uv pip install pycocotools
uv pip install transformers
uv pip install "numpy<2.0.0"
```

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from pycocotools.coco import COCO
from PIL import Image
import os
from transformers import CLIPProcessor, CLIPVisionModel

In [2]:
# Define a placeholder tokenizer for demonstration
# In practice, you would use a tokenizer compatible with nanochat)
class PlaceholderTokenizer:
    def __init__(self):
        self.vocab = {"<SOS>": 1, "<EOS>": 2, "<PAD>": 0}  # Example vocabulary
        self.id_counter = 3

    def encode(self, text):
        tokens = [self.vocab.get(t, 0) for t in text.lower().split()] 
        return [self.vocab['<SOS>']] + tokens + [self.vocab['<EOS>']]

In [12]:
class CocoClipDataset(Dataset):
    def __init__(self, images_dir, ann_file, clip_model_name="openai/clip-vit-base-patch32"):
        self.coco = COCO(ann_file)
        self.images_dir = images_dir
        self.ids = list(self.coco.imgs.keys())
        
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.vision_model = CLIPVisionModel.from_pretrained(clip_model_name).to(self.device)
        self.processor = CLIPProcessor.from_pretrained(clip_model_name)
        
        self.tokenizer = PlaceholderTokenizer()  # Replace with actual tokenizer for nanochat
        
        self.vision_model
        

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]

        img_info = self.coco.loadImgs(img_id)[0]
        img_path = os.path.join(self.images_dir, img_info['file_name'])
        image = Image.open(img_path).convert("RGB")
        
        inputs = self.processor(images=image, return_tensors="pt").to(self.device)
        
        with torch.no_grad():
            vision_outputs = self.vision_model(**inputs)
            image_embeddings = vision_outputs.last_hidden_state
            
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)
        # Each image has 5 captions. Just take first 
        caption = anns[0]['caption']
        caption_tokens = self.tokenizer.encode(caption) 
        
        return {"image_embeddings": image_embeddings.squeeze(0), "caption_tokens": caption_tokens}

In [13]:
IMAGE_DIR = "COCO_data/val2017"
ANN_FILE = "COCO_data/annotations/captions_val2017.json"

if not os.path.exists(IMAGE_DIR) or not os.path.exists(ANN_FILE):
    print("Please update IMAGE_DIR and ANN_FILE with the correct paths to your COCO dataset.")
    
else:
    coco_pipeline = CocoClipDataset(IMAGE_DIR, ANN_FILE)
    sample = coco_pipeline[0]
    
    # print the shapes and types to verify
    print("Image Embeddings Shape:", sample["image_embeddings"].shape)
    print("Caption Tokens:", sample["caption_tokens"])
    
    print(f"type of image_embeddings: {type(sample['image_embeddings'])}")
    print(f"shape of image_embeddings: {sample['image_embeddings'].shape}")

    # Expected shape: [num_patches + 1, embedding_dim] (e.g., [50, 768]) for CLIP ViT-B/32
    # The +1 is for the CLS token
    
    print(f"type of caption_tokens: {type(sample['caption_tokens'])}")
    print(f"shape of caption_tokens: {sample['caption_tokens'].shape if isinstance(sample['caption_tokens'], torch.Tensor) else 'N/A'}")
    # Expected type: list of token IDs (e.g., [1, 5, 10, 2]) where 1 is <SOS>, 2 is <EOS>, and others are word tokens
    
    print(f"content of caption_tokens: {sample['caption_tokens']}")
    

loading annotations into memory...
Done (t=0.04s)
creating index...
index created!


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
visual_projection.weight                                     | UNEXPECTED |  | 
text_model.embeddings.position_embedding.weight              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias      

Image Embeddings Shape: torch.Size([50, 768])
Caption Tokens: [1, 0, 0, 0, 0, 0, 0, 0, 0, 2]
type of image_embeddings: <class 'torch.Tensor'>
shape of image_embeddings: torch.Size([50, 768])
type of caption_tokens: <class 'list'>
shape of caption_tokens: N/A
content of caption_tokens: [1, 0, 0, 0, 0, 0, 0, 0, 0, 2]
